# ⚽ Visión por computadora con YOLO: analizando el Mundial 2026

**Taller práctico** · Chequia 🇨🇿 vs Sudáfrica 🇿🇦 · Atlanta, 18 de junio de 2026 · 79 s de juego continuo

En este notebook vamos a construir paso a paso un sistema que, a partir de un video de la transmisión de TV:

1. **Detecta** jugadores, árbitros y balón con YOLO
2. **Sigue** a cada jugador a lo largo del video (tracking)
3. **Asigna equipo** a cada jugador según el color de su camiseta (K-Means)
4. **Calcula la posesión** del balón por equipo
5. *(Bonus)* Estima el **movimiento de cámara**, convierte píxeles a **metros** y mide **velocidad y distancia** de cada jugador

Basado en [abdullahtarek/football_analysis](https://github.com/abdullahtarek/football_analysis), adaptado a video de transmisión.

> **Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`

## 0. Preparación

In [ ]:
# ⚙️ Configuración del taller (el instructor llena estos valores)
REPO_URL = "https://github.com/JosephLeyva/proxy_football_analysis"   # repo con el código adaptado
CLIP_DRIVE_ID = ""          # ID de Google Drive del clip wc2026_clip.mp4 (vacío = subirlo a mano)
MODEL_DRIVE_ID = "1DC2kCygbBWUKheQ_9cFziCsYVSRw6axK"   # best.pt del repo original

# True = usar los resultados precalculados (stubs) en vez de correr YOLO sobre los 2370 frames
USE_STUBS = True

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !nvidia-smi --query-gpu=name,memory.total --format=csv
    if not os.path.exists("/content/proyecto"):
        !git clone -q {REPO_URL} /content/proyecto
    %cd /content/proyecto
    !pip install -q -r requirements.txt
else:
    # Ejecucion local: el notebook vive en taller/
    if os.path.basename(os.getcwd()) == "taller":
        os.chdir("..")
print("Directorio de trabajo:", os.getcwd())

In [ ]:
import sys, config

if not os.path.exists(config.MODEL_PATH):
    !{sys.executable} -m gdown -q {MODEL_DRIVE_ID} -O {config.MODEL_PATH}

if not os.path.exists(config.VIDEO_PATH):
    if CLIP_DRIVE_ID:
        !{sys.executable} -m gdown -q {CLIP_DRIVE_ID} -O {config.VIDEO_PATH}
    elif IN_COLAB:
        from google.colab import files
        print("Sube el archivo wc2026_clip.mp4")
        uploaded = files.upload()
        os.rename(next(iter(uploaded)), config.VIDEO_PATH)

print("Modelo:", os.path.exists(config.MODEL_PATH), "| Video:", os.path.exists(config.VIDEO_PATH))

In [ ]:
# Utilidades para mostrar imagenes y videos dentro del notebook
import cv2
import numpy as np
import matplotlib.pyplot as plt
from base64 import b64encode
from IPython.display import HTML

def show(*images, titles=None, width=16):
    fig, axes = plt.subplots(1, len(images), figsize=(width, width*9/16/len(images)+1))
    axes = np.atleast_1d(axes)
    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap="gray")
        ax.axis("off")
        if titles:
            ax.set_title(titles[i])
    plt.tight_layout()
    plt.show()

def show_video(path, width=900):
    data = b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

In [ ]:
from utils import read_video, get_video_fps

frames = read_video(config.VIDEO_PATH)
fps = get_video_fps(config.VIDEO_PATH)
print(f"{len(frames)} frames · {fps:.0f} fps · {len(frames)/fps:.0f} s · resolución {frames[0].shape[1]}x{frames[0].shape[0]}")
show(frames[0], titles=["Frame 0"])

> 💡 **Ojo con la memoria:** `read_video` guarda *todos* los frames en RAM. Un frame de 720p pesa ~2.7 MB,
> así que 79 s a 30 fps son ~6.5 GB (a 1080p serían ~15 GB y Colab no aguanta). El partido completo no cabe;
> por eso trabajamos con un clip de una sola toma, sin cortes a repeticiones ni primeros planos.
> YOLO reescala cada imagen a 640 px, así que bajar a 720p no le quita precisión.

## 1. YOLO "de fábrica" (COCO) · ~15 min

YOLO (*You Only Look Once*) mira la imagen **una sola vez** y predice todas las cajas y clases al mismo tiempo.
Los modelos preentrenados de Ultralytics conocen las 80 clases del dataset COCO: `person`, `sports ball`, `car`...

¿Qué tan bien funciona en fútbol sin entrenarlo?

In [ ]:
from ultralytics import YOLO

coco_model = YOLO("yolo11n.pt")          # n = nano, el mas pequeño y rapido
result = coco_model.predict(frames[0], conf=0.25, verbose=False)[0]

from collections import Counter
print(Counter(result.names[int(c)] for c in result.boxes.cls))
show(result.plot(line_width=2, font_size=10), titles=["YOLO11n entrenado con COCO"])

**Observa:**
- Todos son `person`: YOLO no sabe distinguir **jugador, portero o árbitro**.
- El balón (`sports ball`) es diminuto: a veces no lo detecta.
- En otros frames puede detectar gente del público o del cuerpo técnico.

➡️ Necesitamos un modelo **especializado** en fútbol.

🧪 **Reto:** prueba `yolo11x.pt` (el más grande) o baja `conf` a `0.1`. ¿Aparece el balón? ¿Cuánto más tarda?

## 2. Un YOLO especializado en fútbol · ~15 min

`models/best.pt` es un YOLOv5x reentrenado (*fine-tuning*) con el dataset
[football-players-detection de Roboflow](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc):
imágenes de partidos etiquetadas con 4 clases.

Así se entrenó (tarda bastante incluso con GPU, **no lo corras ahora**):

```python
!pip install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="TU_API_KEY")
dataset = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc").version(1).download("yolov5")

!yolo task=detect mode=train model=yolov5x.pt data={dataset.location}/data.yaml epochs=100 imgsz=640
```

In [ ]:
import time

model = YOLO(config.MODEL_PATH)
print("Clases:", model.names)

t = time.time()
result = model.predict(frames[0], conf=0.1, verbose=False)[0]
print(f"Tiempo: {time.time()-t:.2f} s")
print(Counter(result.names[int(c)] for c in result.boxes.cls))
show(result.plot(line_width=2, font_size=10), titles=["best.pt: ball / goalkeeper / player / referee"])

Ahora sí distingue **jugadores, árbitros y balón** 🎉 (¡incluso el juez de línea!). Pero:
- Cada frame es independiente: el jugador de la izquierda en el frame 0 no "sabe" que es el mismo del frame 1.
- A veces confunde clases o detecta cosas fuera de la cancha.

🧪 **Reto:** corre el modelo sobre `frames[300]` y `frames[600]`. ¿Hay algún error de clase?

## 3. Tracking: seguir a cada jugador · ~20 min

El *tracker* (**ByteTrack**) une las detecciones de frames consecutivos: predice hacia dónde se moverá cada caja
y la empareja con la detección más cercana. Así cada jugador conserva un **ID** durante el video.

Además, en `trackers/tracker.py` agregamos dos mejoras para video de TV:
1. **Máscara de cancha**: se descartan detecciones cuyos pies no están sobre el pasto (tribunas, anuncios, marcador).
2. **Voto por mayoría**: si un track fue "jugador" en 95 frames y "árbitro" en 5, se queda como jugador.

In [ ]:
from utils import get_pitch_mask

mask = get_pitch_mask(frames[0])
show(frames[0], mask*255, titles=["Frame", "Máscara de cancha (pasto verde + envolvente convexa): filtro grueso"])

In [ ]:
from trackers import Tracker

tracker = Tracker(config.MODEL_PATH, conf=config.DETECTION_CONF,
                  overlay_boxes=config.OVERLAY_BOXES, ball_control_box=config.BALL_CONTROL_BOX)

# Con USE_STUBS=True se cargan los resultados guardados; si no, corre YOLO + ByteTrack (varios minutos en T4)
tracks = tracker.get_object_tracks(frames, read_from_stub=USE_STUBS, stub_path=config.TRACK_STUB_PATH)
tracker.add_position_to_tracks(tracks)

print("Estructura:", {k: f"{len(v)} frames" for k, v in tracks.items()})
print("Frame 0, jugador de ejemplo:", next(iter(tracks["players"][0].items())))

In [ ]:
# Dibujamos IDs sobre algunos frames (copias, para no modificar los originales)
def draw_tracks(frame_num):
    frame = frames[frame_num].copy()
    for track_id, player in tracks["players"][frame_num].items():
        tracker.draw_ellipse(frame, player["bbox"], (0, 0, 255), track_id)
    for _, referee in tracks["referees"][frame_num].items():
        tracker.draw_ellipse(frame, referee["bbox"], (255, 255, 255))
    return frame

show(draw_tracks(150), draw_tracks(300), titles=["Frame 150", "Frame 300 (5 s después): mismos IDs"])

In [ ]:
unique_ids = {tid for frame in tracks["players"] for tid in frame}
per_frame = [len(f) for f in tracks["players"]]

plt.figure(figsize=(12, 3))
plt.plot(per_frame)
plt.xlabel("frame"); plt.ylabel("jugadores detectados")
plt.title(f"Jugadores por frame · IDs únicos en todo el clip: {len(unique_ids)}")
plt.show()

En la cancha hay 22 jugadores, pero la cámara solo muestra una parte. Si hay **más IDs únicos** que jugadores
que entraron a cuadro, es porque el tracker perdió a alguien (oclusión, salió de cuadro) y le dio un ID nuevo:
es un *ID switch*, el problema clásico del tracking.

🧪 **Reto:** busca un ID que aparezca solo unos pocos frames. ¿Qué pasó ahí?

## 4. ¿De qué equipo es cada jugador? K-Means · ~20 min

Nadie etiquetó los equipos: los vamos a descubrir con **aprendizaje no supervisado**.

**Paso A:** para cada jugador, recortamos la mitad superior de su caja (la camiseta) y agrupamos sus píxeles
en 2 colores con K-Means: uno es el fondo (pasto) y otro la camiseta. El fondo es el color de las esquinas.

In [ ]:
from sklearn.cluster import KMeans

player_id, player = next(iter(tracks["players"][0].items()))
x1, y1, x2, y2 = map(int, player["bbox"])
crop = frames[0][y1:y2, x1:x2]
top_half = crop[: crop.shape[0]//2]

kmeans = KMeans(n_clusters=2, n_init=1).fit(top_half.reshape(-1, 3))
clustered = kmeans.labels_.reshape(top_half.shape[:2])

big = lambda img: cv2.resize(img, None, fx=6, fy=6, interpolation=cv2.INTER_NEAREST)
show(big(crop), big(top_half), big((clustered*255).astype(np.uint8)),
     titles=[f"Jugador {player_id}", "Mitad superior", "2 clusters de color"], width=10)

**Paso B:** juntamos el color de camiseta de todos los jugadores (en 10 frames) y volvemos a usar K-Means con
2 grupos: ¡los equipos!

In [ ]:
from team_assigner import TeamAssigner

team_assigner = TeamAssigner()
team_assigner.assign_team_color(frames, tracks["players"])

sample = [team_assigner.get_player_color(frames[0], p["bbox"]) for p in tracks["players"][0].values()]
sample = np.array(sample)
plt.figure(figsize=(6, 5))
plt.scatter(sample[:, 2], sample[:, 1], c=sample[:, ::-1]/255, s=150, edgecolors="k")
for team, color in team_assigner.team_colors.items():
    plt.scatter(color[2], color[1], marker="X", s=400, c=[color[::-1]/255], edgecolors="k", label=f"Centro equipo {team}")
plt.xlabel("Rojo"); plt.ylabel("Verde"); plt.legend(); plt.title("Color de camiseta de cada jugador (frame 0)")
plt.show()

**Paso C: corregir a YOLO con el color.** El modelo a veces marca a un jugador checo como `referee` durante
cientos de frames (el voto por mayoría de la sección 3 no lo arregla porque el error es consistente).
Pero ya conocemos los colores de los equipos: si la camiseta de un "árbitro" es casi igual a la de un equipo,
es un jugador. Los árbitros (negro) y jueces de línea quedan lejos de ambos colores.

In [ ]:
moved = team_assigner.fix_referees(frames, tracks, max_ratio=config.REFEREE_TEAM_COLOR_RATIO)
print(f"Tracks de 'árbitro' que en realidad son jugadores: {sorted(moved)}")

**Paso D: equipo de cada jugador en cada frame.** Cada 5 frames volvemos a medir el color de la camiseta y el
equipo es la mayoría de las últimas 9 mediciones. ¿Por qué no fijarlo una vez por ID? Porque el tracker a veces
le pasa el ID de un jugador a otro (*ID switch*): el ID 33, por ejemplo, empieza siendo un jugador amarillo y
termina siendo uno rojo.

In [ ]:
for frame_num, player_track in enumerate(tracks["players"]):
    for player_id, track in player_track.items():
        team = team_assigner.get_player_team(frames[frame_num], track["bbox"], player_id)
        track["team"] = team
        track["team_color"] = team_assigner.team_colors[team]

def draw_teams(frame_num):
    frame = frames[frame_num].copy()
    for track_id, player in tracks["players"][frame_num].items():
        tracker.draw_ellipse(frame, player["bbox"], player["team_color"], track_id)
    return frame

show(draw_teams(150), draw_teams(1500), titles=["Frame 150", "Frame 1500"])

🧪 **Retos:**
- ¿Qué pasa con los **porteros**? Su camiseta es de otro color. ¿A qué equipo los asigna? (Lo resolvemos en la sección 6.)
- ¿Funcionaría igual un partido donde ambos equipos visten colores parecidos?

## 5. El balón y la posesión · ~15 min

El balón es pequeño y rápido: YOLO lo pierde en muchos frames. Como su trayectoria es suave,
podemos **interpolar** las posiciones faltantes... pero solo en huecos cortos (< 0.7 s). En un hueco largo el
balón pudo ir a cualquier lado, y una línea recta dibujaría el marcador lejos del balón real.

In [ ]:
import copy
raw_ball = copy.deepcopy(tracks["ball"])   # detecciones originales: las limpiamos en la sección 6

ball_x_raw = [f[1]["bbox"][0] if 1 in f else np.nan for f in tracks["ball"]]
tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"], max_gap=config.BALL_MAX_GAP_FRAMES)
tracker.add_position_to_tracks(tracks)   # recalcular posiciones con el balon interpolado
ball_x = [f[1]["bbox"][0] if 1 in f else np.nan for f in tracks["ball"]]

print(f"Frames sin balón detectado: {np.isnan(ball_x_raw).sum()} de {len(ball_x_raw)}")
plt.figure(figsize=(12, 3))
plt.plot(ball_x, label="interpolado", color="orange")
plt.plot(ball_x_raw, ".", markersize=3, label="detectado por YOLO", color="k")
plt.xlabel("frame"); plt.ylabel("x del balón (px)"); plt.legend(); plt.show()

**Posesión:** en cada frame, el balón es del jugador cuyo pie está más cerca (máximo `config.MAX_PLAYER_BALL_DISTANCE` px).
Si nadie está cerca (pase en el aire), la posesión sigue siendo del último equipo que la tuvo.

In [ ]:
from player_ball_assigner import PlayerBallAssigner

player_assigner = PlayerBallAssigner(max_player_ball_distance=config.MAX_PLAYER_BALL_DISTANCE)

def compute_possession():
    control = []
    for frame_num, player_track in enumerate(tracks["players"]):
        for player in player_track.values():
            player["has_ball"] = False
        ball = tracks["ball"][frame_num].get(1)
        assigned = player_assigner.assign_ball_to_player(player_track, ball["bbox"]) if ball else -1
        if assigned != -1:
            player_track[assigned]["has_ball"] = True
            control.append(player_track[assigned]["team"])
        else:
            control.append(control[-1] if control else 0)
    control = np.array(control)

    for team in (1, 2):
        print(f"Equipo {team}: {np.mean(control[control>0]==team)*100:.1f}% de posesión")
    colors = {t: c[::-1]/255 for t, c in team_assigner.team_colors.items()}
    plt.figure(figsize=(12, 1.5))
    plt.scatter(range(len(control)), np.zeros(len(control)),
                c=[colors.get(t, (0.8, 0.8, 0.8)) for t in control], marker="|", s=800)
    plt.yticks([]); plt.xlabel("frame"); plt.title("¿Quién tiene el balón?"); plt.show()
    return control

team_ball_control = compute_possession()

🧪 **Reto:** cambia `max_player_ball_distance` a 30 y a 150. ¿Cómo cambia la posesión? ¿Qué valor te parece más realista?

## 6. Bonus: de píxeles a metros · ~15 min

Para medir **velocidad** necesitamos metros, no píxeles. Dos problemas:

1. **La cámara se mueve**: en este clip recorre casi toda la cancha y hace zoom. Con *optical flow* seguimos
   cientos de puntos entre frames consecutivos y calculamos la **homografía** que lleva un frame al anterior
   (RANSAC ignora los puntos que se mueven solos, como los jugadores). Encadenándolas, cualquier frame se
   puede llevar al **frame de referencia** (`config.CALIBRATION_FRAME = 150`).
2. **La perspectiva**: 1 px cerca de la cámara no mide lo mismo que 1 px al fondo. En el frame de referencia
   calculamos otra homografía, de píxeles a metros, con puntos conocidos de la cancha: línea de medio campo,
   bandas y círculo central (radio 9.15 m).
3. **Quién está en la cancha**: con la posición en metros quitamos a las personas **detrás de la valla**
   publicitaria (stewards, fotógrafos). La máscara de pasto de la sección 3 no basta, porque YOLO a veces dibuja
   su caja hasta el pasto. En metros es fácil: todo lo que esté a más de 3 m de las líneas, fuera.

In [ ]:
from camera_movement_estimator import CameraMovementEstimator

camera_estimator = CameraMovementEstimator(frames[0], overlay_boxes=config.OVERLAY_BOXES,
                                           text_box=config.CAMERA_MOVEMENT_BOX,
                                           reference_frame=config.CALIBRATION_FRAME)
camera_movement = camera_estimator.get_camera_movement(frames, read_from_stub=USE_STUBS,
                                                       stub_path=config.CAMERA_STUB_PATH)
camera_estimator.add_adjust_positions_to_tracks(tracks, camera_movement)

print("Homografía del frame 2000 al frame de referencia:")
print(camera_estimator.to_reference(camera_movement, 2000).round(3))

summary = camera_estimator.get_camera_summary(camera_movement, frames[0].shape)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3))
ax1.plot(summary[:, 0], label="X"); ax1.plot(summary[:, 1], label="Y")
ax1.set_xlabel("frame"); ax1.set_ylabel("px"); ax1.set_title("Paneo respecto al frame de referencia"); ax1.legend()
ax2.plot(summary[:, 2], color="purple")
ax2.set_xlabel("frame"); ax2.set_title("Zoom respecto al frame de referencia")
plt.show()

In [ ]:
# Frame de referencia: donde se calibró
!{sys.executable} tools/calibrate_pitch.py --check
# Frame 1950: la cámara está en otra zona y con otro zoom; la cancha se dibuja siguiendo las homografías encadenadas
!{sys.executable} tools/calibrate_pitch.py --check --frame 1950 --output output_videos/calibration_check_1950.jpg
show(cv2.imread("output_videos/calibration_check.jpg"), cv2.imread("output_videos/calibration_check_1950.jpg"),
     titles=["Frame 150 (calibración)", "Frame 1950 (siguiendo a la cámara)"])

In [ ]:
from view_transformer import ViewTransformer
from speed_and_distance_estimator import SpeedAndDistance_Estimator

view_transformer = ViewTransformer(config.PITCH_POINTS_PX, config.PITCH_POINTS_M,
                                   config.PITCH_LENGTH_M, config.PITCH_WIDTH_M,
                                   margin=config.PITCH_MARGIN_M)
view_transformer.add_transformed_position_to_tracks(tracks)

# Ya sabemos dónde está cada persona en metros: fuera quien esté a más de 3 m de la cancha
# (stewards, fotógrafos y recogebalones detrás de la valla publicitaria)
print("Detecciones fuera de la cancha eliminadas:", tracker.remove_off_pitch(tracks))

### El balón que no era balón

Mira la gráfica del balón de la sección 5: hay saltos imposibles. YOLO a veces confunde con el balón el
**punto penal** (una mancha blanca en el pasto) o algo en el borde de la imagen, y como nos quedábamos con la
detección de mayor confianza, el marcador saltaba de un lado a otro de la cancha.

Ahora que tenemos metros, usamos la **física**: de todas las candidatas de cada frame nos quedamos con la más
cercana a la última posición del balón, y solo si pudo llegar ahí (un disparo fuerte anda por 30-40 m/s).

In [ ]:
def ball_to_meters(frame_num, point):
    H = camera_estimator.to_reference(camera_movement, frame_num)
    return view_transformer.pixel_to_meters(point, H)

tracks["ball"] = copy.deepcopy(raw_ball)
print("Detecciones descartadas:", tracker.select_ball(tracks, ball_to_meters, fps))
tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"], max_gap=config.BALL_MAX_GAP_FRAMES)
tracker.add_position_to_tracks(tracks)

ball_x_clean = [f[1]["bbox"][0] if 1 in f else np.nan for f in tracks["ball"]]
plt.figure(figsize=(12, 3))
plt.plot(ball_x_raw, ".", markersize=3, color="lightgray", label="YOLO (mayor confianza)")
plt.plot(ball_x_clean, color="orange", label="siguiendo la trayectoria")
plt.xlabel("frame"); plt.ylabel("x del balón (px)"); plt.legend(); plt.show()

### ¿De qué equipo es el portero?

El portero no viste los colores de su equipo, así que K-Means lo manda al equipo con el color más parecido
(a veces el rival). Tampoco basta con mirar a los jugadores cercanos: en un contragolpe o un córner lo rodean rivales.

La idea: responder dos preguntas que **no dependen de un solo frame**:
1. **¿Qué portería defiende cada equipo?** En cada frame, el equipo que defiende la portería derecha suele estar
   más a la derecha que el rival. Los contragolpes invierten unos pocos frames, pero votamos con *todo* el video.
2. **¿De qué lado está cada portero?** YOLO distingue la clase `goalkeeper` (no en todos los frames); guardamos esa marca en el track, descartamos los que visten el color de un equipo y
   tomamos la **mediana** de su posición en metros. Un portero casi nunca sale de su tercio de cancha.

In [ ]:
goalkeeper_teams = team_assigner.assign_goalkeepers(frames, tracks)
confidence = max(team_assigner.side_vote, 1 - team_assigner.side_vote)
print(f"El equipo {team_assigner.team_right} defiende la portería derecha en el {confidence*100:.0f}% de los frames")
print("Porteros (track: equipo):", goalkeeper_teams)

# Con el balón limpio, los porteros corregidos y sin la gente de fuera de la cancha, recalculamos la posesión
team_ball_control = compute_possession()

In [ ]:
speed_estimator = SpeedAndDistance_Estimator(frame_rate=fps, max_speed_kmh=config.MAX_PLAYER_SPEED_KMH)
speed_estimator.add_speed_and_distance_to_tracks(tracks)

In [ ]:
# Vista "radar": posiciones reales en la cancha en un frame
def draw_radar(frame_num, ax):
    ax.set_facecolor("#3a7d44")
    ax.plot([0, 0], [0, 68], "w"); ax.add_patch(plt.Circle((0, 34), 9.15, fill=False, color="w"))
    for team_key in ("players", "referees"):
        for tid, t in tracks[team_key][frame_num].items():
            if t.get("position_transformed") is None:
                continue
            x, y = t["position_transformed"]
            color = t["team_color"][::-1]/255 if "team_color" in t else "white"
            ax.scatter(x, y, color=color, s=120, edgecolors="k")
    ax.set_xlim(-52.5, 52.5); ax.set_ylim(68, 0); ax.set_aspect("equal")
    ax.set_title(f"Frame {frame_num}"); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, n in zip(axes, [150, 1200, 2200]):
    draw_radar(n, ax)
plt.show()

In [ ]:
import pandas as pd

rows = []
for frame in tracks["players"]:
    for tid, t in frame.items():
        if "speed" in t:
            rows.append({"id": tid, "equipo": t["team"], "km/h": t["speed"], "metros": t["distance"]})

stats = pd.DataFrame(rows).groupby("id").agg(equipo=("equipo", "first"),
                                             vel_max=("km/h", "max"),
                                             distancia=("metros", "max"))
stats.sort_values("vel_max", ascending=False).head(10).round(1)

⚠️ **Limitaciones (para discutir):** encadenar homografías acumula error (*drift*): tras miles de frames la cancha
proyectada se desvía unos metros. La solución profesional es detectar los puntos clave de la cancha en *cada*
frame con otro modelo (ver [roboflow/sports](https://github.com/roboflow/sports)).

## 🎬 Video final

In [ ]:
# Dibujamos sobre los frames originales (ya no los necesitamos limpios) y guardamos el video
from utils import save_video

DRAW_IDS = config.DRAW_TRACK_IDS          # True/False: número de cada jugador
DRAW_SPEED = config.DRAW_SPEED_DISTANCE   # True/False: km/h y metros recorridos

output = tracker.draw_annotations(frames, tracks, team_ball_control, team_assigner.team_colors, draw_ids=DRAW_IDS)
output = camera_estimator.draw_camera_movement(output, camera_movement)
if DRAW_SPEED:
    speed_estimator.draw_speed_and_distance(output, tracks)
save_video(output, config.OUTPUT_PATH, fps=fps)
show_video(config.OUTPUT_PATH)

## 7. Retos para seguir · ~10 min

1. **Mapa de calor** de un jugador usando `position_transformed` y `plt.hist2d`.
2. **Radar animado**: convierte `draw_radar` en un video lado a lado con la transmisión.
3. **Otro clip**: usa `tools/find_wide_shots.py` para encontrar otra toma abierta, recórtala y recalibra con
   `tools/calibrate_pitch.py`.
4. **Entrena tu propio modelo** con YOLO11 y el dataset de Roboflow (sección 2) y compáralo con `best.pt`.
5. **Porteros**: asígnalos al equipo cuya mitad de cancha defienden, en vez de usar el color.

### Créditos
- Código base: [abdullahtarek/football_analysis](https://github.com/abdullahtarek/football_analysis) (el autor tiene un tutorial completo en YouTube)
- Dataset: [Roboflow football-players-detection](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc)
- Modelos y librerías: [Ultralytics YOLO](https://docs.ultralytics.com), [supervision](https://supervision.roboflow.com)
- Video: transmisión de DSports, Copa Mundial de la FIFA 2026 (uso educativo)